In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import Counter


In [ ]:
#Params

url = "https://api.openalex.org/works"

params = {
    "filter": (
        "primary_topic.subfield.id:subfields/1203,"
        "type:types/article,"
        "primary_topic.id:t10034,"
        "publication_year:2015-2024"
    ),
    "sort": "cited_by_count:desc",
    "per_page": 200
}

headers = {
    "User-Agent": "frederico.prado@proton.me"
}

langlist = pd.read_csv("linglist.csv")


In [72]:
#Fetch data

all_results = []
cursor = "*"

while True:
    print("Fetching… cursor:", cursor)
    
    params["cursor"] = cursor
    
    resp = requests.get(url, params=params, headers=headers)
    resp.raise_for_status()
    data = resp.json()
    
    results = data.get("results", [])
    if not results:
        break
    
    all_results.extend(results)
    
    cursor = data.get("meta", {}).get("next_cursor")
    if not cursor:
        break

print("\nFetched", len(all_results), "works total.")


Fetching… cursor: *
Fetching… cursor: Ils3NSwgMTAwLjAsIDc1LCAnaHR0cHM6Ly9vcGVuYWxleC5vcmcvVzI1NjA0OTk2NzYnXSI=
Fetching… cursor: Ils1MCwgMTAwLjAsIDUwLCAnaHR0cHM6Ly9vcGVuYWxleC5vcmcvVzI5NDQ3MDE5NDAnXSI=
Fetching… cursor: IlszOCwgMTAwLjAsIDM4LCAnaHR0cHM6Ly9vcGVuYWxleC5vcmcvVzE4NTY0ODA3NTQnXSI=
Fetching… cursor: IlszMCwgMTAwLjAsIDMwLCAnaHR0cHM6Ly9vcGVuYWxleC5vcmcvVzIyODEwMzgyNzInXSI=
Fetching… cursor: IlsyNiwgOTguMCwgMjYsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMjE4OTU3NzU5NSddIg==
Fetching… cursor: IlsyMiwgOTkuMCwgMjIsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMjU5MDUwMTUyOCddIg==
Fetching… cursor: IlsxOSwgOTguMCwgMTksICdodHRwczovL29wZW5hbGV4Lm9yZy9XMjE2NzgzNDU2NiddIg==
Fetching… cursor: IlsxNiwgOTguMCwgMTYsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMjUyMTAxOTY0MCddIg==
Fetching… cursor: IlsxMywgMTAwLjAsIDEzLCAnaHR0cHM6Ly9vcGVuYWxleC5vcmcvVzQzOTUwODc2MzUnXSI=
Fetching… cursor: IlsxMSwgOTguMCwgMTEsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMjczODUyNjk3NCddIg==
Fetching… cursor: IlsxMCwgOTYuMCwgMTAsICdodHRwczovL29wZW5hbGV4Lm9yZy9X

In [76]:
def top_concept(work):
    concepts = work.get("concepts", [])
    if not concepts:
        return None
    return max(concepts, key=lambda c: c["score"])["display_name"]

def first_author_country(work):
    for auth in work.get("authorships", []):
        for inst in auth.get("institutions", []):
            if inst.get("country"):
                return inst["country"]
    return None


In [59]:
languages = (
    langlist["language"]
    .dropna()
    .drop_duplicates()
    .str.strip()
    .tolist()
)


lang_pattern = re.compile(
    r"\b(" + "|".join(re.escape(lang) for lang in languages) + r")\b",
    flags=re.IGNORECASE
)


In [ ]:
rows = []

for w in all_results:
    rows.append({
        "title": w.get("title"),
        "year": w.get("publication_year"),
        "doi": w.get("doi"),
        "cited_by": w.get("cited_by_count"),
        "journal": w.get("host_venue", {}).get("display_name"),
        "primary_topic": w.get("primary_topic", {}).get("display_name"),
        "concepts": [c["display_name"] for c in w.get("concepts", [])],
        "top_concept": top_concept(w),
        "paper_country": first_author_country(w)
    })

df = pd.DataFrame(rows)
df.head()


In [61]:
df["language_mentioned"] = df.apply(detect_languages, axis=1)

df[["title", "language_mentioned"]].head(15)


,title,language_mentioned
0,Semantic Interpretation in Generative Grammar,None
1,Transitivity in Grammar and Discourse,None
2,A theory of focus interpretation,None
3,The Semantics of Definite and Indefinite Noun ...,None
4,Rules and Representations.,None
5,Regularity and Idiomaticity in Grammatical Con...,None
6,The Mental Representation of Grammatical Relat...,None
7,Rethinking Linguistic Relativity,None
8,From Usage to Grammar: The Mind's Response to ...,None
9,Three Factors in Language Design,None


In [62]:
df_lang = df[df["language_mentioned"].notna()]
df_lang.head()

all_langs = [
    lang.strip()
    for cell in df_lang["language_mentioned"]
    for lang in cell.split(",")
]

lang_counts = pd.DataFrame(
    Counter(all_langs).most_common(20),
    columns=["language", "count"]
)

lang_counts


,language,count
0,English,305
1,German,182
2,Bantu languages,131
3,Chinese,103
4,Japanese,84
5,Spanish,71
6,Icelandic,66
7,Romance languages,54
8,Germanic languages,53
9,French,52


In [64]:
paper_countries = []

for w in all_results:
    country = None
    
    # Loop through authors in order
    for auth in w.get("authorships", []):
        # Loop through that author's institutions
        for inst in auth.get("institutions", []):
            if inst.get("country"):
                country = inst["country"]
                break
        if country:
            break
    
    paper_countries.append({
        "title": w.get("title"),
        "paper_country": country
    })

df_country = pd.DataFrame(paper_countries)

# Merge into your main dataframe
df = df.merge(df_country, on="title", how="left")

# Check result
df[["title", "paper_country"]].head(10)

,title,paper_country
0,Semantic Interpretation in Generative Grammar,None
1,Transitivity in Grammar and Discourse,None
2,Transitivity in Grammar and Discourse,None
3,A theory of focus interpretation,None
4,The Semantics of Definite and Indefinite Noun ...,None
5,Rules and Representations.,None
6,Regularity and Idiomaticity in Grammatical Con...,None
7,The Mental Representation of Grammatical Relat...,None
8,Rethinking Linguistic Relativity,None
9,Rethinking Linguistic Relativity,None
